In [1]:
import importlib
import copy
import time
import traceback
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from Experiment.potentials import get_potential_energy_CH3CN, get_potential_energy_CH3CN_harmonic
from Experiment.utils import get_orbitals_indices_first, get_energy_clusters, get_ttno
from Experiment.utils_ch3cn import random_mps_harmonic_oscillator_0, random_threetree_harmonic_oscillator_0, random_leafonly_harmonic_oscillator_0
from CheTTNS_A_B import *
from pytreenet.ttns import TreeTensorNetworkState
from pytreenet.ttno import TreeTensorNetworkOperator
from pytreenet.operators.models.two_site_model import XYModel
from pytreenet.operators.common_operators import *
from pytreenet.dmrg import DMRGAlgorithm
from pytreenet.ttns.ttns_ttno.application import apply_ttno_to_ttns, ApplicationMethod
from pytreenet.util import SVDParameters
from pytreenet.util.misc_functions import linear_combination
from Dipole_moment import get_mu_ttno_linear
import utils_CH3CN as ch3cn_utils
from pytreenet.util.misc_functions import add
from pytreenet.core.truncation import truncate_ttns, TruncationMethod

ch3cn_utils = importlib.reload(ch3cn_utils)
CH3CN_IR_FREQ_CM_12 = ch3cn_utils.CH3CN_IR_FREQ_CM_12
CH3CN_IR_TRANSITION_DIPOLE_12 = ch3cn_utils.CH3CN_IR_TRANSITION_DIPOLE_12
CH3CN_ORCA_FUNDAMENTAL_FREQ_CM_12 = ch3cn_utils.CH3CN_ORCA_FUNDAMENTAL_FREQ_CM_12
get_ch3cn_ir_dipole_derivative = ch3cn_utils.get_ch3cn_ir_dipole_derivative
get_ch3cn_orca_potential_energy = ch3cn_utils.get_ch3cn_orca_potential_energy


[cell 0 elapsed: 1.28s]


### CH3CN TTNO

This uses the same CH3CN Hamiltonian construction cell as `CH3CN_compare_10dim_150N.ipynb`, then only rescales/shifts the TTNO.


In [3]:
from ttno_shift_utils import (
    identity_ttno_like,
    subtract_identity_shift,
    scale_ttno_with_energy_window_shifted,
)

N = [9, 7, 9, 9, 9, 9, 7, 7, 9, 9, 27, 27]
node_order = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

ORCA_FORCE_CONSTANT_CUTOFF_CM = 1.0

omega_experiment, _, _ = get_potential_energy_CH3CN_harmonic()
omega, k3_orca, k4_orca = get_ch3cn_orca_potential_energy(
    force_constant_cutoff_cm=ORCA_FORCE_CONSTANT_CUTOFF_CM,
)

_, orb_state, orb_Es = get_orbitals_indices_first(omega_experiment, num_orb=1)

state0 = random_threetree_harmonic_oscillator_0(
    N,
    omega_experiment,
    orb_state[0].reshape(1, -1),
    node_order,
)
state0.canonical_form(state0.root_id)

ttno, _ = get_ttno(N, state0, get_potential_energy_CH3CN, True)

site_ids = list(state0.nodes.keys())
print("Number of sites:", len(site_ids))
print("Site ids:", site_ids)
print("Experiment harmonic omega cm^-1:", np.asarray(omega_experiment) * 1000.0)
print("ORCA harmonic omega cm^-1:", omega * 1000.0)
print("ORCA k3 nonzero:", int(np.count_nonzero(k3_orca)))
print("ORCA k4 nonzero:", int(np.count_nonzero(k4_orca)))
print("orb_E:", orb_Es[0])
print("Original TTNO bond dims:", ttno.bond_dims())


Number of sites: 13
Site ids: ['site12', 'site3', 'site2', 'site1', 'site0', 'site4', 'site5', 'site6', 'site7', 'site8', 'site9', 'site10', 'site11']
Experiment harmonic omega cm^-1: [3065. 2297. 1413.  920. 3149. 3149. 1487. 1487. 1061. 1061.  361.  361.]
ORCA harmonic omega cm^-1: [3048.42 2364.22 1413.32  929.02 3116.15 3116.39 1473.16 1473.19 1062.64
 1062.8   381.75  381.8 ]
ORCA k3 nonzero: 215
ORCA k4 nonzero: 338
orb_E: 9.9055
Original TTNO bond dims: {('site12', 'site3'): 14, ('site3', 'site2'): 11, ('site2', 'site1'): 9, ('site1', 'site0'): 5, ('site12', 'site4'): 18, ('site4', 'site5'): 16, ('site5', 'site6'): 10, ('site6', 'site7'): 5, ('site12', 'site8'): 18, ('site8', 'site9'): 13, ('site9', 'site10'): 9, ('site10', 'site11'): 5}
[cell 2 elapsed: 0.30s]


### Scale And Subtract Identity Shift

`CH3CN_compare_10dim_150N.ipynb` printed these DMRG window values, so this notebook skips the DMRG min/max step and only tests the TTNO operation.


In [5]:
def scale_ttno_with_energy_window_no_shift(
    hamiltonian,
    E_min,
    E_max,
    W_prime=0.9875,
    safety_factor=1.10,
):
    if not np.isfinite(E_min) or not np.isfinite(E_max) or E_max <= E_min:
        raise ValueError(f"Invalid energy window: E_min={E_min}, E_max={E_max}")
    if safety_factor < 1.0:
        raise ValueError(f"safety_factor must be >= 1, got {safety_factor}")

    a = safety_factor * (E_max - E_min) / (2.0 * W_prime)
    H_scaled = copy.deepcopy(hamiltonian)
    H_scaled.tensors[H_scaled.root_id] = H_scaled.tensors[H_scaled.root_id] * (1.0 / a)
    shift = (E_min / a) + W_prime
    return H_scaled, float(a), float(shift)


CH3CN_REFERENCE_E_MIN = 9.85838834931253
CH3CN_REFERENCE_E_MAX = 290.2947399795622
RESCALE_W_PRIME = 0.9875
RESCALE_SAFETY_FACTOR = 1.10

H_scaled_only, a_old, shift_old = scale_ttno_with_energy_window_no_shift(
    ttno,
    E_min=CH3CN_REFERENCE_E_MIN,
    E_max=CH3CN_REFERENCE_E_MAX,
    W_prime=RESCALE_W_PRIME,
    safety_factor=RESCALE_SAFETY_FACTOR,
)

H_scaled_shifted, a_new, shift_new = scale_ttno_with_energy_window_shifted(
    ttno,
    E_min=CH3CN_REFERENCE_E_MIN,
    E_max=CH3CN_REFERENCE_E_MAX,
    W_prime=RESCALE_W_PRIME,
    safety_factor=RESCALE_SAFETY_FACTOR,
)

H_scaled_manual_shifted = subtract_identity_shift(H_scaled_only, shift_old)

print("a_old:", a_old)
print("a_new:", a_new)
print("shift_old:", shift_old)
print("shift_new:", shift_new)
print("same a:", np.allclose(a_old, a_new))
print("same shift:", np.allclose(shift_old, shift_new))
print("Scaled-only TTNO bond dims:", H_scaled_only.bond_dims())
print("Shifted TTNO bond dims:", H_scaled_shifted.bond_dims())


a_old: 156.1923983763416
a_new: 156.1923983763416
shift_old: 1.050616953525222
shift_new: 1.050616953525222
same a: True
same shift: True
Scaled-only TTNO bond dims: {('site12', 'site3'): 14, ('site3', 'site2'): 11, ('site2', 'site1'): 9, ('site1', 'site0'): 5, ('site12', 'site4'): 18, ('site4', 'site5'): 16, ('site5', 'site6'): 10, ('site6', 'site7'): 5, ('site12', 'site8'): 18, ('site8', 'site9'): 13, ('site9', 'site10'): 9, ('site10', 'site11'): 5}
Shifted TTNO bond dims: {('site12', 'site3'): 15, ('site12', 'site4'): 19, ('site12', 'site8'): 19, ('site3', 'site2'): 12, ('site2', 'site1'): 10, ('site1', 'site0'): 6, ('site4', 'site5'): 17, ('site5', 'site6'): 11, ('site6', 'site7'): 6, ('site8', 'site9'): 14, ('site9', 'site10'): 10, ('site10', 'site11'): 6}
[cell 4 elapsed: 0.00s]


### Subtract Function Check On `state0`

For any state, the shifted operator should satisfy `<psi|H_scaled - shift I|psi> = <psi|H_scaled|psi> - shift <psi|psi>`.


In [7]:
state_norm = state0.scalar_product()
identity_ttno = identity_ttno_like(ttno)
identity_expectation = state0.operator_expectation_value(identity_ttno)

scaled_expectation = state0.operator_expectation_value(H_scaled_only)
shifted_expectation = state0.operator_expectation_value(H_scaled_shifted)
manual_shifted_expectation = state0.operator_expectation_value(H_scaled_manual_shifted)
expected_shifted_expectation = scaled_expectation - shift_old * state_norm

print("state_norm:", state_norm)
print("<state0|I|state0>:", identity_expectation)
print("<state0|H_scaled|state0>:", scaled_expectation)
print("expected <state0|(H_scaled - shift I)|state0>:", expected_shifted_expectation)
print("new function <state0|H_shifted|state0>:", shifted_expectation)
print("manual subtract <state0|H_shifted|state0>:", manual_shifted_expectation)
print("identity abs error:", abs(identity_expectation - state_norm))
print("new shifted abs error:", abs(shifted_expectation - expected_shifted_expectation))
print("manual-vs-new abs error:", abs(manual_shifted_expectation - shifted_expectation))

assert np.allclose(identity_expectation, state_norm, rtol=1e-8, atol=1e-8)
assert np.allclose(shifted_expectation, expected_shifted_expectation, rtol=1e-8, atol=1e-8)
assert np.allclose(manual_shifted_expectation, shifted_expectation, rtol=1e-8, atol=1e-8)


state_norm: (1+0j)
<state0|I|state0>: (1.0000000000000004+0j)
<state0|H_scaled|state0>: (0.06407288033241268+0j)
expected <state0|(H_scaled - shift I)|state0>: (-0.9865440731928095+0j)
new function <state0|H_shifted|state0>: (-0.98654407319281+0j)
manual subtract <state0|H_shifted|state0>: (-0.98654407319281+0j)
identity abs error: 4.440892098500626e-16
new shifted abs error: 5.551115123125783e-16
manual-vs-new abs error: 0.0
[cell 6 elapsed: 0.00s]
